The following script performs EEG data preprocessing through several steps:
1. Read raw file
2. Band pass filter 1 to 45hz
3. Crop signal from tmin to tmax
4. Visual inspection of channels. Drop bads
5. Epochs of -0.3s to 1.2s
6. Autoreject Epochs
7. Manual inspection of Epochs
8. ICA
9. Interpolate bad channels
10. Rereferenced to grand average

In [1]:
# This magic command allows interactive plotting in a separate window
# %matplotlib qt

# Import necessary libraries for the preprocessing
import os
os.environ['GIT_PYTHON_REFRESH'] = 'quiet'  # Suppress git errors
from git import Repo
import numpy as np

import matplotlib.pyplot as plt

import mne
from mne_bids import BIDSPath, read_raw_bids #,print_dir_tree
# Importing libraries for automatic rejection of bad epochs
from autoreject import AutoReject, get_rejection_threshold
from pyprep import NoisyChannels


# tag automatically ICA components
# requires pytorch
from mne_icalabel import label_components

# Import helper functions for preprocessing
from utils.log_preprocessing import LogPreprocessingDetails
from utils import bids_compliance


# 1. Load Data

In [2]:
def read_raw_custom(subject, date, task, root='DATA', data_name='data'):
    """
    Reads raw EEG data from a custom directory structure similar to BIDS.

    Parameters:
    - subject (str): Subject identifier (e.g., 'S002')
    - date (str): Date of the recording in 'YYYY-MM-DD' format
    - task (str): Task identifier (e.g., 'Sart1')
    - root (str): Root directory of the data (default is 'DATA')
    - data_name (str): Name of the data variable in the .mat file (default is 'data')

    Returns:
    - raw (mne.io.Raw): The raw EEG data
    """
    # Construct the filename based on the provided parameters
    filename = f'CYBERSART_{date}_{task}_eeg.mat'
    filepath = os.path.join(root, f'sub-{subject}', 'eeg', filename)
    
    # Create an MNE Info object
    # You'll need to specify the sampling frequency, channel names, and types
    # For this example, we'll use placeholder values
    sfreq = 1000  # Replace with your actual sampling frequency
    ch_names = ['Fp1', 'Fp2', 'F3', 'F4', 'C3', 'C4', 'P3', 'P4', 'O1', 'O2']  # Replace with your actual channel names
    ch_types = ['eeg'] * len(ch_names)
    info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types=ch_types)
    
    # Read the data using mne.io.read_raw_fieldtrip
    raw = mne.io.read_raw_fieldtrip(filepath, info, data_name=data_name)
    
    return raw

# Get the current working directory
cwd = os.getcwd()

# Assuming the script is run from within the repository
repo = Repo(os.getcwd(), search_parent_directories=True)
repo_root = repo.git.rev_parse("--show-toplevel")

# Define the file path components
results_folder = "DATA"
root =os.path.join(repo_root, results_folder)
# Example usage:
subject = 'S002'
date = '2019-11-07'
task = 'Sart1'  # Could be 'Sart1', 'Sart2', etc.
raw = read_raw_custom(subject, date, task, root=root)

GitCommandNotFound: Cmd('git') not found due to: FileNotFoundError('[WinError 2] Le fichier spécifié est introuvable')
  cmdline: git rev-parse --show-toplevel

In [7]:
##################################
#####          LOAD          #####
##################################
# Get the current working directory
cwd = os.getcwd()

# Assuming the script is run from within the repository
repo = Repo(os.getcwd(), search_parent_directories=True)
repo_root = repo.git.rev_parse("--show-toplevel")

# Define the file path components
results_folder = "DATA"
# print_dir_tree(os.path.join(repo_root, results_folder,return_str = False))

subject = "11"
session = "a"
task = "sartauditiva"  #'sartvisual' #'narrative'
data = "eeg"

# Create a BIDSPath object
bids_path = BIDSPath(
    subject=subject,
    session=session,
    task=task,
    datatype=data,
    suffix=data,
    extension=".vhdr",
    root=os.path.join(repo_root, results_folder),
)

##################################
#####      FOR SAVING        #####
##################################
# Defining the paths for saving results and raw data
derivatives_folder = os.path.join(repo_root, "derivatives")
bids_dir = os.path.join(derivatives_folder, f"sub-{subject}", f"ses-{session}", "eeg")
os.makedirs(bids_dir, exist_ok=True)

# Initialize a report to document the preprocessing steps
report = mne.Report(
    title=f"Preprocessing sub-{subject} for session {session} and {task}"
)

# Path to the JSON file where preprocessing details will be stored
json_path = os.path.join(
    repo_root, derivatives_folder, "logs_preprocessing_details_all_subjects_eeg.json"
)

# Initialize the logging class
log_preprocessing = LogPreprocessingDetails(json_path, subject, session, task)


##################################
########   1.READ RAW   ##########
##################################

# Read Raw bids
raw = read_raw_bids(bids_path)

# import bad chs from another task of same session
# Important to check if also bad!!!
raw.info["bads"] = log_preprocessing.import_bad_channels_another_task()

print(raw.info)

# Plot sensor location in the scalp
# raw.plot_sensors(show_names=True)
# plt.show()

# Add the raw data info to the report
report.add_raw(raw=raw, title="Raw", psd=True)

# Log the raw data info
log_preprocessing.log_detail("info", str(raw.info))

Embedding : jquery-3.6.0.min.js
Embedding : bootstrap.bundle.min.js
Embedding : bootstrap.min.css
Embedding : bootstrap-table/bootstrap-table.min.js
Embedding : bootstrap-table/bootstrap-table.min.css
Embedding : bootstrap-table/bootstrap-table-copy-rows.min.js
Embedding : bootstrap-table/bootstrap-table-export.min.js
Embedding : bootstrap-table/tableExport.min.js
Embedding : bootstrap-icons/bootstrap-icons.mne.min.css
Embedding : highlightjs/highlight.min.js
Embedding : highlightjs/atom-one-dark-reasonable.min.css
Extracting parameters from C:\Users\nicolas.bruno\Documents\GitHub\wandering-mind\results\sub-11\ses-a\eeg\sub-11_ses-a_task-sartauditiva_eeg.vhdr...
Setting channel info structure...
Reading events from C:\Users\nicolas.bruno\Documents\GitHub\wandering-mind\results\sub-11\ses-a\eeg\sub-11_ses-a_task-sartauditiva_events.tsv.
Reading channel info from C:\Users\nicolas.bruno\Documents\GitHub\wandering-mind\results\sub-11\ses-a\eeg\sub-11_ses-a_task-sartauditiva_channels.tsv.
R

C:\Users\nicolas.bruno\AppData\Local\Temp\ipykernel_30568\3199868454.py:58: RuntimeWarning: The unit for channel(s) GSR has changed from NA to S.
  raw = read_raw_bids(bids_path)
C:\Users\nicolas.bruno\AppData\Local\Temp\ipykernel_30568\3199868454.py:58: RuntimeWarning: There are channels without locations (n/a) that are not marked as bad: ['ECG', 'R_EYE', 'GSR', 'RESP']
  raw = read_raw_bids(bids_path)
C:\Users\nicolas.bruno\AppData\Local\Temp\ipykernel_30568\3199868454.py:58: RuntimeWarning: Not setting positions of 4 ecg/eog/gsr/resp channels found in montage:
['ECG', 'R_EYE', 'GSR', 'RESP']
Consider setting the channel types to be of EEG/sEEG/ECoG/DBS/fNIRS using inst.set_channel_types before calling inst.set_montage, or omit these channels when creating your montage.
  raw = read_raw_bids(bids_path)

A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.1.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some m

Channels marked as bad:
none
Attempting to create new mne-python configuration file:
C:\Users\nicolas.bruno\.mne\mne-python.json
Effective window size : 4.096 (s)
Plotting power spectral density (dB=True).


# 2.FILTERING

In [23]:
# Apply a band-pass filter to keep frequencies between 1 and 45 Hz
hpass = 0.5
lpass = 45
raw_filtered = raw.load_data().copy().notch_filter(np.arange(50, 250, 50)).filter(l_freq=hpass, h_freq=lpass)

# Save the filtered data
# bids_path.update(root = derivatives_folder, description = 'filtered')
# write_raw_bids(raw_filtered, bids_path, format='BrainVision', allow_preload=True, overwrite=True)

# Log the filter settings
log_preprocessing.log_detail("hpass_filter", hpass)
log_preprocessing.log_detail("lpass_filter", lpass)
log_preprocessing.log_detail("filter_type", "bandpass")

Filtering raw data in 1 contiguous segment
Setting up band-stop filter

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandstop filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower transition bandwidth: 0.50 Hz
- Upper transition bandwidth: 0.50 Hz
- Filter length: 3301 samples (6.602 s)



[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.1s


Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.5 - 45 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 0.50
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 0.25 Hz)
- Upper passband edge: 45.00 Hz
- Upper transition bandwidth: 11.25 Hz (-6 dB cutoff frequency: 50.62 Hz)
- Filter length: 3301 samples (6.602 s)



[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.1s


# 3.Visual inspection of CHs

In [24]:
# Plots PSD of the raw data
raw_filtered.compute_psd().plot()

#automatically mark bad channels
nd = NoisyChannels(raw_filtered,do_detrend = False, random_state=42)
nd.find_all_bads(ransac=True, channel_wise=True) #if it slows down, set channel_wise to False
bads = nd.get_bads()
print(f"Bad channels detected: {bads}")
if bads != None:
    raw_filtered.info["bads"] = bads

# Plot the filtered data for visual inspection to identify bad channels
raw_filtered.plot(n_channels=32)
plt.show(block=True)

# Add the filtered data to the report
report.add_raw(raw=raw_filtered, title="Filtered Raw", psd=True)

# Log the identified bad channels
log_preprocessing.log_detail("bad_channels", raw_filtered.info["bads"])

NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Executing RANSAC
This may take a while, so be patient...
Finding optimal chunk size : 30
Total # of chunks: 1
Current chunk:
1

RANSAC done!
Bad channels detected: ['FC1']
Effective window size : 4.096 (s)
Plotting power spectral density (dB=True).
Channels marked as bad:
['FC1']
Channels marked as bad:
['FC1']
Effective window size : 4.096 (s)
Plotting power spectral density (dB=True).


# 4. EPOCHING

In [5]:
##################################
######    LOAD TRIGGGERS   #######
##################################
# Filter annotations by description
filtered_annotations = mne.Annotations(onset=[], duration=[], description=[])

for ann in raw_filtered.annotations:
    if "go/" in ann["description"] or "nogo/" in ann["description"]:
        filtered_annotations.append(ann["onset"], ann["duration"], ann["description"])

raw_filtered.set_annotations(filtered_annotations)

events, event_id = mne.events_from_annotations(raw_filtered)

# Segment the continuous data into epochs of 2 seconds
tmin = -0.3
tmax = 1.2
# baseline correction should be done after ICA
epochs = mne.Epochs(
    raw_filtered,
    events=events,
    event_id=event_id,
    tmin=tmin,
    tmax=tmax,
    preload=False,
    verbose=False,
)

# Save the epoched data
# bids_compliance.save_epoched_bids(epochs, derivatives_folder, subject, session,
#                                   task, data, desc = 'epoched', events = events, event_id =event_id)

# Add the epochs to the report
report.add_epochs(epochs=epochs, title="Epochs")

# Log the number of epochs and their duration
log_preprocessing.log_detail("n_epochs", len(epochs))
log_preprocessing.log_detail("tmin", tmin)
log_preprocessing.log_detail("tmax", tmax)

Used Annotations descriptions: ['go/correct/segment-0/10,26/distracted/completely confident/somewhat immersed', 'go/correct/segment-0/11,25/distracted/completely confident/somewhat immersed', 'go/correct/segment-0/12,24/distracted/completely confident/somewhat immersed', 'go/correct/segment-0/13,23/distracted/completely confident/somewhat immersed', 'go/correct/segment-0/14,22/distracted/completely confident/somewhat immersed', 'go/correct/segment-0/15,21/distracted/completely confident/somewhat immersed', 'go/correct/segment-0/17,19/distracted/completely confident/somewhat immersed', 'go/correct/segment-0/18,18/distracted/completely confident/somewhat immersed', 'go/correct/segment-0/19,17/distracted/completely confident/somewhat immersed', 'go/correct/segment-0/2,34/distracted/completely confident/somewhat immersed', 'go/correct/segment-0/21,15/distracted/completely confident/somewhat immersed', 'go/correct/segment-0/22,14/distracted/completely confident/somewhat immersed', 'go/corre

# 5. Reject Bad Epochs 1

In [ ]:
# TODO: add rejection for acceloremeter (available from sub 14 onwards)
folds=5 # increase for more accuracy or decrease for speed
# Automatically reject bad epochs using AutoReject
ar = AutoReject(thresh_method="bayesian_optimization", cv = folds, random_state=42, n_jobs = -1, )
epochs_clean = ar.fit_transform(epochs)
reject = get_rejection_threshold(epochs)

ar.get_reject_log(epochs).plot('horizontal')

# Log the epochs rejected by AutoReject
ar_reject_epochs = [
    n_epoch
    for n_epoch, log in enumerate(epochs_clean.drop_log)
    if log == ("AUTOREJECT",)
]

log_preprocessing.log_detail("autoreject_epochs", ar_reject_epochs)
log_preprocessing.log_detail("autoreject_threshold", reject)
log_preprocessing.log_detail("len_autoreject_epochs", len(ar_reject_epochs))

In [ ]:
# epochs_clean = epochs  # to skip autoreject
# ar_reject_epochs = []  # to skip autoreject
#Manually inspect and reject bad epochs
epochs_clean.plot(n_channels=32)
plt.show(block=True)

# Log the epochs rejected manually
manual_reject_epochs = [
    n_epoch for n_epoch, log in enumerate(epochs_clean.drop_log) if log == ("USER",)
]
print(f"Manually rejected epochs: {manual_reject_epochs}")
total_epochs_rejected = (
    (len(ar_reject_epochs) + len(manual_reject_epochs)) / len(epochs) * 100
)
print(f"Total epochs rejected: {total_epochs_rejected}%")
log_preprocessing.log_detail("manual_reject_epochs", manual_reject_epochs)
log_preprocessing.log_detail("len_manual_reject_epochs", len(manual_reject_epochs))

# Plot the drop log for further inspection
epochs_clean.plot_drop_log()

# Add the cleaned epochs to the report
report.add_epochs(epochs=epochs_clean, title="Epochs clean", psd=False)

# Save the cleaned epochs
epochs_clean.drop_bad()
# bids_compliance.save_epoched_bids(epochs_clean, derivatives_folder, subject, session,
#                                   task, data, desc = 'epochedClean', events = events, event_id =event_id)

# 6. Independent Component Analysis (ICA)

In [ ]:
# Parameters for ICA (Independent Component Analysis) to remove artifacts
n_components = 15
method = "picard"  # The algorithm to use for ICA
max_iter = (
    "auto"  # Maximum number of iterations; typically should be higher, like 500 or 1000
)
random_state = 42  # Seed for random number generator for reproducibility

# Initialize the ICA object with the specified parameters
ica = mne.preprocessing.ICA(
    n_components=n_components,
    method=method,
    max_iter=max_iter,
    random_state=random_state,
)

# Fit the ICA model to the cleaned epochs
ica.fit(epochs_clean)

# find EOG artifacts in the data via pattern matching, and exclude the EOG-related ICA components
eog_components, eog_scores = ica.find_bads_eog(
    inst=epochs_clean,
    ch_name="R_EYE",  # a channel close to the eye
    # threshold=1  # lower than the default threshold
)
print(f"EOG components detected: {eog_components}")

# find ECG artifacts in the data via pattern matching, and exclude the ECG-related ICA components
ecg_components, ecg_scores = ica.find_bads_ecg(
    inst=epochs_clean,
    ch_name="ECG",  # a channel close to the eye
    # threshold=1  # lower than the default threshold
)
print(f"ECG components detected: {ecg_components}")

# find muscle artifacts in the data via pattern matching, and exclude the muscle-related ICA components
muscle_components, muscle_scores = ica.find_bads_muscle(epochs_clean, threshold=0.7)
print(f"Muscle components detected: {muscle_components}")
# ica.plot_scores(muscle_scores, exclude=muscle_components)

# Combine all artifact components from the pattern matching methods
pattern_matching_artifacts = np.unique(ecg_components + eog_components + muscle_components)

##### Classify the components using ICLabel model #######
# run the model on the ICA components
ic_labels = label_components(epochs_clean, ica, method="iclabel")
# print labels of each component
print("Classification of all ICA components. Results:")
print(ic_labels["labels"])

# Extract ICA component labels
label_names = ic_labels['labels']

# Identify the ICA components that correspond to a 'channel noise' in ICLabel
channel_artifact_indices = [i for i, label in enumerate(label_names) if label == 'channel noise']

# Find components that coincide between pattern matching and ICLabel output for exclusion
# We'll only exclude components that match the artifacts found via pattern matching 
# and are classified as 'muscle artifact', 'eye blink', 'heart beat', or 'channel noise'
to_exclude = []
for idx in pattern_matching_artifacts:
    if label_names[idx] in ['muscle artifact', 'eye blink', 'heart beat', 'channel noise']:
        to_exclude.append(idx)

# Also ensure to include 'channel noise' components that were found only by ICLabel
to_exclude = np.unique(to_exclude + channel_artifact_indices)

# Exclude the selected components
ica.exclude = to_exclude.tolist()

# (Optional) Plot the ICA components for visual inspection
# ica.plot_components(inst=epochs_clean, picks=range(15))

# Plot the sources identified by ICA
ica.plot_sources(epochs_clean, block=True, show=True)
plt.show(block=True)

# Add the ICA results to the report
report.add_ica(ica, title="ICA", inst=epochs_clean)

# Apply the ICA solution to the cleaned epochs
epochs_ica = ica.apply(inst=epochs_clean)

# Log the ICA parameters and excluded components
log_preprocessing.log_detail("ica_components", ica.exclude)
log_preprocessing.log_detail("ica_method", method)
log_preprocessing.log_detail("ica_max_iter", max_iter)
log_preprocessing.log_detail("ica_random_state", random_state)

In [ ]:
##### FINAL EPOCH CLEANING #######
baseline = (-0.3, 0)  # to be done after ICA!
epochs_ica.apply_baseline(baseline)
log_preprocessing.log_detail("baseline", baseline)

# Manually inspect the epochs after ICA application
epochs_ica.plot(n_channels=32)
plt.show(block=True)

# Log manually rejected epochs after ICA
all_manual_epochs = [
    n_epoch for n_epoch, log in enumerate(epochs_ica.drop_log) if log == ("USER",)
]
manual_reject_epochs_after_ica = [
    n_epoch for n_epoch in all_manual_epochs if n_epoch not in manual_reject_epochs
]
print(f"Manually rejected epochs after ICA: {manual_reject_epochs_after_ica}")
total_epochs_rejected = (
(len(ar_reject_epochs)+len(manual_reject_epochs)+len(manual_reject_epochs_after_ica)
    )/ len(epochs) * 100
)
print(f"Total epochs rejected: {total_epochs_rejected}%")
log_preprocessing.log_detail("manual_reject_epochs_after_ica", manual_reject_epochs_after_ica)
log_preprocessing.log_detail("len_manual_reject_epochs_after_ica", len(manual_reject_epochs_after_ica))
log_preprocessing.log_detail("total_epochs_rejected", total_epochs_rejected)
log_preprocessing.log_detail("epochs_drop_log", epochs_ica.drop_log)
log_preprocessing.log_detail("epochs_drop_log_description", epochs_ica.drop_log)

# Save the epochs after ICA application and drop epochs
# bids_compliance.save_epoched_bids(epochs_ica, derivatives_folder, subject, session,
#                                   task, data, desc = 'epochedICA', events = events, event_id =event_id)

# 7. Interpolate Chs and Rereference

In [ ]:
##################################
######   Interpolate chs  ########
##################################
# Interpolate bad channels in the epochs after ICA application
epochs_interpolate = epochs_ica.copy().interpolate_bads()

# Log the interpolated channels
log_preprocessing.log_detail("interpolated_channels", epochs_ica.info["bads"])

##################################
#######    Rereference   #########
##################################
# Rereference the data to the grand average reference
epochs_rereferenced, ref_data = mne.set_eeg_reference(
    inst=epochs_interpolate, ref_channels="average", copy=True
)

# Add the final epochs to the report
report.add_epochs(
    epochs=epochs_rereferenced, title="Epochs interpolated and rereferenced", psd=True
)

# Log the rereferencing details
log_preprocessing.log_detail("rereferenced_channels", "grand_average")

# SAVE Preprocessed data 

In [ ]:
# Save the rereferenced epochs
bids_compliance.save_epoched_bids(
    epochs_rereferenced,
    derivatives_folder,
    subject,
    session,
    task,
    data,
    desc="preproc",
    events=events,
    event_id=event_id,
)

# Save the report as an HTML file
html_report_fname = bids_compliance.make_bids_basename(
    subject=subject,
    session=session,
    task=task,
    suffix=data,
    extension=".html",
    desc="preprocReport",
)
report.save(os.path.join(bids_dir, html_report_fname), overwrite=True)

# Save the preprocessing details to the JSON file
log_preprocessing.save_preprocessing_details()

Optional: Observe Preprocessed Data 

In [ ]:
epochs_rereferenced.plot()
plt.show(block=True)